### Construção das redes (TVG) e extração de features

Este notebook gera as redes de conectividade funcional a partir dos EEG pré-processados.

Fluxo:
1. Carrega `.set` em `data/preprocessed` (opcional: só amostra saudável, mesmo critério de `02`/`03`)
2. Motifs → threshold (tese: shuffle da série por eletrodo → motifs → sync de todos os pares → p99) → TVG
   - Recorte: 8 s contínuos (2000 amostras a 250 Hz) → 1998 motifs de ordem 3
   - Janela de 25 motifs (100 ms), deslocada de 1 em 1 → 1967 janelas
   - Atraso de até 6 amostras (24 ms) **nos dois sentidos** (i antecede j e j antecede i), comparando só pares dentro da janela (25 − τ comparações); a sincronização do par é o máximo entre os atrasos
3. Extrai métricas de rede
4. Salva features em `data/intermediate/sub-*_*.parquet`
5. Salva 3 CSV estilo Rosário em `data/intermediate/networks/` (`*_nodes`, `*_temporal`, `*_edges`)

Arquivos intermediários já existentes são pulados. Com `ONLY_HEALTHY=True`, só entram sujeitos do filtro saudável.

Próximo passo: `03_realign_features_by_electrode.ipynb` (realinha por eletrodo e regenera o parquet processado).


### Imports

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mne
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.graph.aggregate import (
    aggregate_static_network,
    clustering_over_time,
    degree_over_time,
    edges_per_time,
    temporal_statistics,
    window_node_features,
)
from src.graph.export_networks import rea_from_exports, save_network_exports
from src.graph.motifs import trans_motifs
from src.graph.network import graph_metrics
from src.graph.eeg_crop import crop_continuous_segment
from src.graph.threshold import estimate_threshold
from src.graph.tvg import build_tvg_parallel


### Paths

In [2]:
SOURCE_DIR = Path("../../data/preprocessed")
OUTPUT_DIR = Path("../../data/intermediate")
NETWORKS_DIR = OUTPUT_DIR / "networks"
WINDOWS_DIR = OUTPUT_DIR / "windows"
METADATA_FILE = Path("../../data/metadata/participants.csv")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NETWORKS_DIR.mkdir(parents=True, exist_ok=True)
WINDOWS_DIR.mkdir(parents=True, exist_ok=True)

# Mesmo critério de amostra saudável de 02_build_dataset / 03_realign
ONLY_HEALTHY = True

print("SOURCE_DIR:", SOURCE_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("NETWORKS_DIR:", NETWORKS_DIR.resolve())
print("METADATA_FILE:", METADATA_FILE.resolve())
print("ONLY_HEALTHY:", ONLY_HEALTHY)


SOURCE_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\preprocessed
OUTPUT_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate
NETWORKS_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate\networks
METADATA_FILE: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\metadata\participants.csv
ONLY_HEALTHY: True


### Parâmetros

In [3]:
WINDOW_MS = 100
MAX_LAG_MS = 25
THRESHOLD_PERCENTILE = 99.0

# Recorte contínuo comum (ver notebook 00_network_parameters / tvg_common_continuous_summary.json)
COMMON_JSON = Path("../../data/intermediate/tvg_common_continuous_summary.json")
_common = json.loads(COMMON_JSON.read_text(encoding="utf-8"))
TVG_T0 = float(_common["t0"])
TVG_T1 = float(_common["t1"])
N_SEC = int(_common.get("N_SEC_sugerido", 8))
print(
    f"Recorte TVG: N_SEC={N_SEC} s dentro de [{TVG_T0:.3f}, {TVG_T1:.3f}] s "
    f"(de {COMMON_JSON.name})"
)


Recorte TVG: N_SEC=8 s dentro de [60.002, 68.274] s (de tvg_common_continuous_summary.json)


### Listar arquivos EEG

Com `ONLY_HEALTHY=True`, restringe aos sujeitos do filtro saudável (drug/smoking/audit/hamilton/bsl23/skid), alinhado a `02_build_dataset` / `03_realign`. Log: N saudáveis, arquivos candidatos, já existentes e pendentes.

In [4]:
all_files = sorted(SOURCE_DIR.glob("sub-*.set"))
print(f"Pré-processados: {len(all_files)} arquivos em {SOURCE_DIR}")

if ONLY_HEALTHY:
    meta = pd.read_csv(METADATA_FILE)
    meta = meta.rename(
        columns={
            "ID": "subject_id",
            "DRUG_0=negative_1=Positive": "drug",
            "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
            "SKID_Diagnoses": "skid_diagnoses",
            "SKID_Diagnoses 2": "skid_diagnoses_2",
            "Comments_SKID_assessment": "comments_skid_assessment",
            "Hamilton_Scale": "hamilton_scale",
            "BSL23_behavior": "bsl23_behavior",
            "AUDIT": "audit",
        }
    )
    healthy_mask = (
        (meta["drug"] == 0)
        & (meta["smoking"] == 1)
        & (meta["audit"] < 8)
        & (meta["hamilton_scale"] <= 7)
        & (meta["bsl23_behavior"] == 0)
        & (meta["skid_diagnoses"].str.lower() == "none")
        & (meta["skid_diagnoses_2"].isna())
        & (meta["comments_skid_assessment"].isna())
    )
    healthy_ids = set(meta.loc[healthy_mask, "subject_id"].astype(str))
    files = [
        f for f in all_files
        if f.stem.split("_", 1)[0] in healthy_ids
    ]
    print(f"Sujeitos saudáveis (metadados): {len(healthy_ids)}")
    print(f"Arquivos após filtro saudável: {len(files)} (de {len(all_files)})")
else:
    healthy_ids = None
    files = all_files
    print(f"ONLY_HEALTHY=False → usando todos os {len(files)} arquivos")

def _outputs_exist(stem: str) -> bool:
    return (
        (OUTPUT_DIR / f"{stem}.parquet").exists()
        and (NETWORKS_DIR / f"{stem}_nodes.csv").exists()
        and (NETWORKS_DIR / f"{stem}_temporal.csv").exists()
        and (NETWORKS_DIR / f"{stem}_edges.csv").exists()
    )

n_exist = sum(1 for f in files if _outputs_exist(f.stem))
n_pending = len(files) - n_exist
print(f"Já existentes (skip): {n_exist}")
print(f"Pendentes: {n_pending}")
files[:5]

Pré-processados: 315 arquivos em ..\..\data\preprocessed
Sujeitos saudáveis (metadados): 66
Arquivos após filtro saudável: 108 (de 315)
Já existentes (skip): 108
Pendentes: 0


[WindowsPath('../../data/preprocessed/sub-010069_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010069_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010070_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010070_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010079_EC.set')]

### Processar arquivos (saudáveis / pendentes)

Para cada `.set` da lista filtrada: motifs → threshold → TVG → métricas → parquet em `data/intermediate`.
Arquivos com parquet + 3 CSVs já existentes são pulados.


In [5]:
for file in tqdm(files, desc="Arquivos"):
    subject_id, condition = file.stem.split("_")
    stem = f"{subject_id}_{condition}"
    out_file = OUTPUT_DIR / f"{stem}.parquet"
    nodes_csv = NETWORKS_DIR / f"{stem}_nodes.csv"
    temporal_csv = NETWORKS_DIR / f"{stem}_temporal.csv"
    edges_csv = NETWORKS_DIR / f"{stem}_edges.csv"

    if out_file.exists() and nodes_csv.exists() and temporal_csv.exists() and edges_csv.exists():
        print(f"Já existe: {stem} (pulando)")
        continue

    # ==========================
    # LOAD EEG
    # ==========================
    print(f"\nProcessando {file.name}...")
    raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
    raw.pick(picks="eeg")
    ch_names = list(raw.ch_names)

    eeg, sfreq, start_samp, stop_samp = crop_continuous_segment(
        raw,
        TVG_T0,
        TVG_T1,
        N_SEC,
        file_name=file.name,
    )
    print(
        f"  Recorte contínuo: amostras [{start_samp}, {stop_samp}) "
        f"≈ [{start_samp/sfreq:.3f}, {stop_samp/sfreq:.3f}] s"
    )

    window_size = int(WINDOW_MS / 1000 * sfreq)
    max_lag = int(MAX_LAG_MS / 1000 * sfreq)

    # ==========================
    # MOTIFS + THRESHOLD + TVG
    # ==========================
    # Limiar (tese): embaralha série de cada eletrodo → motifs → sync todos os pares → p99
    threshold_info = estimate_threshold(
        eeg,
        window_size,
        max_lag,
        percentile=THRESHOLD_PERCENTILE,
    )
    threshold = threshold_info["threshold"]
    print(
        f"Threshold (p{THRESHOLD_PERCENTILE:.0f}): {threshold:.4f} "
        f"(nulo: μ={threshold_info['null_mean']:.4f}, "
        f"σ={threshold_info['null_std']:.4f}, "
        f"arestas_nulas={threshold_info['n_null_edges']})"
    )

    motifs = trans_motifs(eeg)

    tvg = build_tvg_parallel(
        motifs=motifs,
        window_size=window_size,
        max_lag=max_lag,
        threshold=threshold,
    )

    # ==========================
    # MÉTRICAS
    # ==========================
    print("Calculando métricas de rede...")
    rea = aggregate_static_network(tvg)
    edges_t = edges_per_time(tvg)
    edges_stats = temporal_statistics(edges_t)
    net = graph_metrics(rea, n_windows=tvg.shape[2])
    win = window_node_features(tvg)

    features = {}
    for metric in ("degree", "clustering", "betweenness"):
        for i, v in enumerate(net[metric]):
            features[f"{metric}_{i}"] = v
    for metric, values in win.items():
        for i, v in enumerate(values):
            features[f"{metric}_{i}"] = v

    features["sync_threshold"] = threshold
    features["sync_threshold_null_mean"] = threshold_info["null_mean"]
    features["sync_threshold_null_std"] = threshold_info["null_std"]
    features["edges_mean"] = edges_stats["mean"]
    features["edges_std"] = edges_stats["std"]
    features["edges_cv"] = edges_stats["cv"]

    # ==========================
    # SALVAR FEATURES + REDES
    # ==========================
    df = pd.DataFrame(features, index=[0])
    df["subject_id"] = subject_id
    df["condition"] = condition
    df.to_parquet(out_file, index=False)

    np.savez_compressed(
        WINDOWS_DIR / f"{stem}.npz",
        degree=degree_over_time(tvg).astype(np.int16),
        clustering=clustering_over_time(tvg).astype(np.float32),
        ch_names=np.array(ch_names),
    )

    paths = save_network_exports(
        tvg=tvg,
        rea=rea,
        ch_names=ch_names,
        networks_dir=NETWORKS_DIR,
        stem=stem,
    )
    print(
        f"Salvo: {out_file.name} | TVG shape: {tvg.shape} | "
        f"redes: {paths['nodes'].name}, {paths['temporal'].name}, {paths['edges'].name}"
    )

print("Concluído.")


Arquivos: 100%|██████████| 108/108 [00:00<00:00, 3247.23it/s]

Já existe: sub-010069_EC (pulando)
Já existe: sub-010069_EO (pulando)
Já existe: sub-010070_EC (pulando)
Já existe: sub-010070_EO (pulando)
Já existe: sub-010079_EC (pulando)
Já existe: sub-010079_EO (pulando)
Já existe: sub-010084_EC (pulando)
Já existe: sub-010084_EO (pulando)
Já existe: sub-010086_EC (pulando)
Já existe: sub-010086_EO (pulando)
Já existe: sub-010089_EC (pulando)
Já existe: sub-010089_EO (pulando)
Já existe: sub-010091_EC (pulando)
Já existe: sub-010091_EO (pulando)
Já existe: sub-010092_EC (pulando)
Já existe: sub-010092_EO (pulando)
Já existe: sub-010126_EC (pulando)
Já existe: sub-010126_EO (pulando)
Já existe: sub-010136_EC (pulando)
Já existe: sub-010136_EO (pulando)
Já existe: sub-010137_EC (pulando)
Já existe: sub-010137_EO (pulando)
Já existe: sub-010138_EC (pulando)
Já existe: sub-010138_EO (pulando)
Já existe: sub-010146_EC (pulando)
Já existe: sub-010146_EO (pulando)
Já existe: sub-010157_EC (pulando)
Já existe: sub-010157_EO (pulando)
Já existe: sub-01016

### Métricas da rede agregada (REA)

A REA soma o TVG binário: o peso de cada par é o número de janelas em que ele esteve sincronizado. Ela é completa em todos os registros (todo par sincroniza em pelo menos uma janela), então:

- **Grau (`degree_`)** é o grau ponderado (força), dividido por `n_janelas × (N − 1)`: fração média do tempo em que o eletrodo esteve conectado aos demais, entre 0 e 1.
- **Agrupamento (`clustering_`)** é o ponderado do NetworkX (pesos divididos pelo maior peso da rede): mede a intensidade relativa dos triângulos, não a presença deles.
- **Intermediação (`betweenness_`)** usa distância = 1 / peso, para que pares que sincronizam mais fiquem mais próximos.
- Grau binário e eficiência global/local não são calculados: valem o mesmo para todos numa rede completa. `weighted_degree` também saiu, por ser idêntico ao grau.

A célula abaixo atualiza essas métricas nos intermediários já existentes, a partir de `*_nodes.csv`, `*_edges.csv` e `*_temporal.csv`, sem refazer o TVG. Rode uma vez depois de mudar `graph_metrics`; ela é idempotente.

In [6]:
DROP_PREFIXES = ("weighted_degree_", "local_efficiency_")
DROP_COLS = {"global_efficiency"}

updated = 0
for out_file in tqdm(sorted(OUTPUT_DIR.glob("sub-*.parquet")), desc="Atualizando métricas"):
    stem = out_file.stem
    nodes_csv = NETWORKS_DIR / f"{stem}_nodes.csv"
    edges_csv = NETWORKS_DIR / f"{stem}_edges.csv"
    temporal_csv = NETWORKS_DIR / f"{stem}_temporal.csv"
    if not (nodes_csv.exists() and edges_csv.exists() and temporal_csv.exists()):
        continue

    rea, _ = rea_from_exports(nodes_csv, edges_csv)
    n_windows = len(pd.read_csv(temporal_csv))
    net = graph_metrics(rea, n_windows=n_windows)

    df = pd.read_parquet(out_file)
    df = df.drop(columns=[c for c in df.columns if c.startswith(DROP_PREFIXES) or c in DROP_COLS])
    for metric in ("degree", "clustering", "betweenness"):
        for i, v in enumerate(net[metric]):
            df[f"{metric}_{i}"] = v
    df.to_parquet(out_file, index=False)
    updated += 1

print(f"Intermediários atualizados: {updated}")

Atualizando métricas: 100%|██████████| 236/236 [01:20<00:00,  2.94it/s]

Intermediários atualizados: 236


### Métricas por janela (TVG binário)

Em cada janela o TVG é uma rede binária esparsa, onde grau, agrupamento e hub têm o sentido clássico:

- **Grau por janela** `k_i(t) / (N − 1)`. A média no tempo é igual ao grau da REA normalizado (`degree_`); o desvio padrão no tempo vira `sd_degree_`.
- **Agrupamento binário** `bin_clustering_`: fração dos pares de vizinhos de i que também estão sincronizados entre si, na janela; média no tempo, ignorando janelas em que o nó tem grau < 2.
- **Hub** `hub_frequency_`: em cada janela, o nó é hub quando o grau passa a média da rede + 2 desvios padrão (entre os nós daquela janela); a feature é a fração das janelas em que ele foi hub.

As séries completas (grau e agrupamento por nó e janela) ficam em `data/intermediate/windows/{stem}.npz`, para as figuras com todos os pontos.

A célula abaixo refaz o TVG dos registros já processados usando o limiar salvo no parquet (a REA refeita é conferida contra `*_edges.csv`) e acrescenta essas colunas. Registros já atualizados são pulados.

In [7]:
for file in tqdm(files, desc="Métricas por janela"):
    stem = file.stem
    out_file = OUTPUT_DIR / f"{stem}.parquet"
    windows_file = WINDOWS_DIR / f"{stem}.npz"
    if not out_file.exists():
        continue
    df = pd.read_parquet(out_file)
    if windows_file.exists() and "sd_degree_0" in df.columns:
        continue

    raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
    raw.pick(picks="eeg")
    ch_names = list(raw.ch_names)
    eeg, sfreq, _, _ = crop_continuous_segment(raw, TVG_T0, TVG_T1, N_SEC, file_name=file.name)

    tvg = build_tvg_parallel(
        motifs=trans_motifs(eeg),
        window_size=int(WINDOW_MS / 1000 * sfreq),
        max_lag=int(MAX_LAG_MS / 1000 * sfreq),
        threshold=float(df["sync_threshold"].iloc[0]),
    )
    rea_saved, _ = rea_from_exports(NETWORKS_DIR / f"{stem}_nodes.csv", NETWORKS_DIR / f"{stem}_edges.csv")
    if not np.array_equal(aggregate_static_network(tvg), rea_saved):
        raise ValueError(f"{stem}: TVG refeito não reproduz a REA salva")

    for metric, values in window_node_features(tvg).items():
        for i, v in enumerate(values):
            df[f"{metric}_{i}"] = v
    df.to_parquet(out_file, index=False)

    np.savez_compressed(
        windows_file,
        degree=degree_over_time(tvg).astype(np.int16),
        clustering=clustering_over_time(tvg).astype(np.float32),
        ch_names=np.array(ch_names),
    )

print("Concluído.")

Métricas por janela: 100%|██████████| 108/108 [00:05<00:00, 20.46it/s]

Concluído.


### Próximo passo

Com os intermediários prontos, rode:

- Notebook: `notebook/00_redes/03_realign_features_by_electrode.ipynb`
- (opcional) revisar merge/filtro em `02_build_dataset.ipynb`

Isso gera `data/processed/dataset_features.parquet` para os notebooks de EDA.

Os CSV em `data/intermediate/networks/` (`*_nodes`, `*_temporal`, `*_edges`) são exports estilo Rosário/Gephi; o realinhamento usa `*_nodes` + parquets intermediários.
